# Overtilpasning og kontrollerte eksperimenter

**Du skal kunne:** lese treningskurver, bruke en baseline og sammenligne forsøk
uten å la sluttesten bli en del av treningen. Et stort nett kan pugge et lite
øvingshefte. Dropout er som å øve med tilfeldige deler av arbeidslaget borte,
slik at modellen må finne flere måter å løse oppgaven på.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## Bruk de samme dataene hver gang

Vi bruker det samme tilfeldige skillet som i forrige notatbok, men bare 256
treningsbilder for å gjøre overtilpasning lettere å se. Valideringen har 1000 andre
bilder. Utvalgene overlapper ikke. `lag_lastere` lager de samme DataLoader-typene
som du nettopp har brukt; kildekoden står i `startkode/kursverktoy.py`.


In [ ]:
from torch import nn
from torch.utils.data import Subset, random_split
from torchvision import datasets, transforms
from kursverktoy import lag_lastere, tren, maal
alle = datasets.FashionMNIST(kursmappe / "data", train=True, download=True,
                            transform=transforms.ToTensor())
trening, validering = random_split(alle, [50000, 10000],
                                  generator=torch.Generator().manual_seed(42))
trening = Subset(trening, range(256))
validering = Subset(validering, range(1000))


## En enkel målestokk

En **baseline** er et enkelt sammenligningspunkt. Her finner `bincount` den vanligste
kategorien blant treningsbildene. Baseline gjetter denne kategorien hver gang.
Vi velger kategorien fra trening og måler treff på validering.

**Forutsi:** Hvorfor er «gjetter alltid bukse» bedre enn ingen sammenligning,
selv om vi forventer at nettverket slår den?


In [ ]:
treningsfasit = torch.tensor([int(trening[i][1]) for i in range(len(trening))])
vanligste = torch.bincount(treningsfasit, minlength=10).argmax().item()
baseline = sum(int(validering[i][1]) == vanligste for i in range(len(validering))) / len(validering)
print(f"Baseline på validering: {baseline:.1%}, alltid klasse {vanligste}")


## Tre forsøk med én endring om gangen

Vi bruker samme arkitektur og seed, men endrer først læringsraten, deretter dropout.
`Dropout(0.4)` setter tilfeldig 40 % av aktiveringene til null i treningsmodus;
i evalueringsmodus er dropout slått av. Det er ingen garanti for at dropout hjelper
med akkurat dette utvalget.

`tren` er løkken fra notatbok 02 samlet i en funksjon. Åpne kildefilen og finn
`zero_grad`, `backward` og `step` før kjøring. Funksjonen registrerer kurver, måler
tid per treningsepoke, kopierer beste vekter etter **lavest valideringsfeil**, og
laster dem tilbake til slutt. Derfor er modellen etterpå ikke nødvendigvis fra
siste epoke. Vi beholder en CPU-kopi av de valgte vektene for hvert forsøk.

`sett_seed` nullstiller vektene. Nye DataLoadere får en egen generator med seed 42
for stokkingen, slik at dropout ikke endrer neste epokes datastokking.
Dette gjør sammenligningen mer kontrollert; flere seeds er nødvendig for sikrere
konklusjoner. Et enkelt forsøk er først og fremst et læringseksempel.


In [ ]:
forsok = [("utgangspunkt", 0.001, 0.0),
          ("lavere læringsrate", 0.0001, 0.0),
          ("dropout", 0.001, 0.4)]
resultater, vekter = {}, {}
for navn, lr, dropout in forsok:
    sett_seed(42)
    trening_laster, val_laster = lag_lastere(trening, validering)
    modell = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                           nn.Dropout(dropout), nn.Linear(256, 10)).to(enhet)
    print("\n", navn)
    resultater[navn] = tren(modell, trening_laster, val_laster, enhet,
                            epoker=80, lr=lr)
    vekter[navn] = {k: v.detach().cpu().clone() for k, v in modell.state_dict().items()}


## Se etter et gap

Hvis treningsfeilen faller mens valideringsfeilen stiger over tid, har vi et tegn
på overtilpasning. En forskjell mellom kurvene alene er ikke et bevis. Med dropout
er treningsoppgaven dessuten annerledes fordi deler av nettet er slått av.
`subplots` gir ett panel per forsøk; lik y-akse gjør panelene lettere å sammenligne.


In [ ]:
fig, akser = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for akse, (navn, rader) in zip(akser, resultater.items()):
    akse.plot([r["treningsfeil"] for r in rader], label="Trening")
    akse.plot([r["valideringsfeil"] for r in rader], label="Validering")
    akse.set_title(navn)
    akse.set_xlabel("Epokeindeks fra 0")
    akse.legend()
akser[0].set_ylabel("Cross entropy")
plt.tight_layout()
plt.show()


## Ta valget før sluttesten

Vi velger forsøket med lavest oppnådd valideringsfeil og gjenskaper samme arkitektur.
`state_dict` er navngitte vekter og buffere; det inneholder ikke selve modellklassen.
Så bruker vi det offisielle testsettet én gang. Resultatet er et anslag for denne
oppgaven og fordelingen, ikke en garanti for bilder tatt med eget kamera.

**Stopp:** Skriv ned forventet testresultat og valgt modell før neste celle.
Hvis du fortsetter å endre oppsettet etter testresultatet, har du begynt å bruke
testen til modellvalg. Da trenger du et nytt, urørt evalueringsgrunnlag.


In [ ]:
valgt = min(resultater, key=lambda navn: min(r["valideringsfeil"] for r in resultater[navn]))
dropout = next(d for navn, lr, d in forsok if navn == valgt)
sluttmodell = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                            nn.Dropout(dropout), nn.Linear(256, 10)).to(enhet)
sluttmodell.load_state_dict(vekter[valgt])
test = datasets.FashionMNIST(kursmappe / "data", train=False, download=True,
                            transform=transforms.ToTensor())
from torch.utils.data import DataLoader
test_feil, test_treff = maal(sluttmodell, DataLoader(test, batch_size=128), enhet)
print(f"Valgt: {valgt}. Sluttest: feil={test_feil:.3f}, treff={test_treff:.1%}")


## Din forsøkslogg

Bruk [forsøksloggen](../FORSOKSLOGG.md) i din egen arbeidsmappe. Noter hypotese,
datautvalg, seed, innstillinger, maskin, tidsbruk og hva kurvene faktisk viser.
Prøv deretter et større treningsutvalg som et nytt undervisningsforsøk, og vurder
valideringen. Det gamle testresultatet hører til det forrige forsøket.

**Kontroll:** Hvorfor kan en modell med lavest treningsfeil være et dårlig valg?
**Svar:** Den kan ha pugget øvingsbildene og fungere dårligere på nye bilder.

**Neste:** [04-cnn-og-gpu.ipynb](04-cnn-og-gpu.ipynb).
